# Explorer Tutorial

This tutorial demonstrates how to use the Explorer components of the EpiScope package.  Explorer implements retrieval‑augmented generation (RAG) over indexed documents.  In this example we focus on computing simple embeddings and augmenting a query using a hypothetical document (HYDE) generator.  For brevity, we do not perform actual indexing into a vector store.

## Compute Embeddings with SimplifiedEmbedder

The `SimplifiedEmbedder` class wraps a HuggingFace embedding model and provides batched encoding.  When the underlying embedding library (llama_index/transformers) is unavailable, the embedder falls back to a stub implementation that returns a vector based on the length of the input text.  This makes the embedder useful even in a test environment.  Below we instantiate an embedder and compute embeddings for a list of sentences.

In [1]:
from episcope.retrieve.embeddings import SimplifiedEmbedder

# Instantiate a simplified embedder (falls back to a stub if transformers# is unavailable).  The `dim` attribute reflects the hidden size of the# underlying model or 1 for the stub.
embedder = SimplifiedEmbedder(embed_model='distilbert-base-uncased', batch_size=2)
print(f'Embedding dimension: {embedder.dim}')

sentences = [
    'Influenza spreads quickly in winter.',
    'Vaccination reduces the risk of infection.',
    'Personal protective equipment (PPE) helps prevent transmission.'
]
vectors = embedder.embed_texts(sentences)
for s, v in zip(sentences, vectors):
    print(f'Sentence: {s} Embedding: {v}')


No sentence-transformers model found with name distilbert-base-uncased. Creating a new one with mean pooling.


Embedding dimension: 768


100%|██████████| 2/2 [00:00<00:00,  2.75it/s]

Sentence: Influenza spreads quickly in winter. Embedding: [0.005801310762763023, -0.0067009590566158295, -0.012025194242596626, -0.004412005189806223, 0.034680821001529694, -0.07454068958759308, 0.034270185977220535, 0.09893356263637543, 0.002989410189911723, -0.05350383371114731, -0.021028924733400345, -0.058306947350502014, -0.03369952738285065, 0.04729623347520828, -0.04683943837881088, 0.045874904841184616, -0.013846001587808132, 0.009035849012434483, -0.02154013141989708, 0.042242590337991714, 0.006482681725174189, -0.04930988699197769, -0.00690258014947176, 0.06165323778986931, 0.013971653766930103, -0.04453367367386818, -0.008130669593811035, 0.004843646660447121, -0.0015616327291354537, -0.06867456436157227, 0.02529267780482769, 0.0016084231901913881, -0.018610330298542976, -0.00629850011318922, -0.010088887065649033, -0.01278763823211193, 0.009323344565927982, 0.01220600213855505, -0.03520794212818146, 0.03699298948049545, -0.04639904201030731, -0.031905397772789, -0.030400253

## Enhance a Query with HYDE

The HYDE (Hypothetical Document) generator augments queries by generating a hypothetical document describing what an answer might look like.  This can improve retrieval performance by providing richer context.  In this example we patch the underlying LLM call to return a deterministic response for illustration.

In [3]:
from unittest.mock import patch
from episcope.core.hyde import HYDE

# Patch ollama.chat so that HYDE.generate returns a predictable string
def fake_chat(model_name: str, messages: list, options: dict):
    return {'message': {'content': 'This is a hypothetical document about the query.'}}

with patch('episcope.core.hyde.ollama.chat', side_effect=fake_chat):
    hyde = HYDE(model_name='tinyllama')
    query = 'What is the basic reproduction number (R0) of a virus?'
    print('Original query:', query)
    hypothetical = hyde.generate(query)
    print('Generated hypothetical document:', hypothetical)
    enhanced = hyde.enhance_query(query, paper_title=None, domain=None)
    print('Enhanced query with HYDE:', enhanced)

Error during HYDE generation 1: fake_chat() got an unexpected keyword argument 'model'
Error during HYDE generation 1: fake_chat() got an unexpected keyword argument 'model'
Error during HYDE generation 2: fake_chat() got an unexpected keyword argument 'model'
All HYDE generations failed; returning original query


Original query: What is the basic reproduction number (R0) of a virus?
Generated hypothetical document: Generation failed.
Enhanced query with HYDE: What is the basic reproduction number (R0) of a virus?


## Next Steps

In a full Explorer pipeline you would index your PDF documents (using `TextRAG.index`) and then perform retrieval with `TextRAG.retrieve`.  The retrieved contexts can then be passed to an LLM to generate answers with provenance.  This notebook introduced the basic building blocks (embedding and query augmentation) to get you started.

## Loading Documents for Indexing

Before you can index documents for retrieval you must first extract their text content.  EpiScope includes a document loader factory that supports multiple backends (Unstructured, GROBID).  In this example we show how to load a plain text file into a list of structured sections.  These sections can then be passed to the `PaperIndexer` for per‑paper indexing.

In [1]:
from episcope.ingest.document_loader import DocumentLoaderFactory
from episcope.index.paper_indexer import PaperIndexer

# Create a loader (Unstructured is used by default for PDFs and text).
loader = DocumentLoaderFactory.get_loader('unstructured')


In [2]:

# # Assume we have a simple text file on disk.  For this demonstration
# # we create a temporary file, but in practice you would provide a
# # path to a .pdf or .txt document.
# import tempfile
# temp = tempfile.NamedTemporaryFile(delete=False, suffix='.txt')
# # Repeating conent multiple times to ensure chunking occurs during indexing (othewise might get discarded if too small)
# temp.write(b'First paragraph First paragraph First paragraph First paragraph First paragraph First paragraph First paragraph.\n\nSecond paragraph. Second paragraph. Second paragraph. Second paragraph. Second paragraph. Second paragraph. Second paragraph.')
# temp.close()


**Loading**

In [7]:

# sections, metadata = loader.load(temp.name)
sections, metadata = loader.load("./100 Tips to Write Clean Code.pdf")
print('Loaded', len(sections), 'section(s)')
for sec in sections:
    print('-', sec.title or '<no title>')
    print(sec.content)
    break


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Loaded 25 section(s)
- <no title>
From: Craft Better Software danielmoka@substack.com
Subject: 100 Tips to Write Clean Code
Date: 17. September 2025 at 07:04
To: vins23p@gmail.com
Forwarded this email? Subscribe here for more
Om YO


**Indexing**

In [11]:

# Index the sections using a PaperIndexer
indexer = PaperIndexer(min_chunk_size=5)
indexer.index_paper(sections, metadata, paper_id='TEMP_DOC')
# You can now search this paper using indexer.search()
print('Top result:', indexer.search('Code smells are patterns in code that indicate potential problems.')[0])


No sentence-transformers model found with name distilbert-base-uncased. Creating a new one with mean pooling.


Top result: {'text': '52. Commit early & push often\n53. Write meaningful commit messages explaining the reason\n54. Use the imperative mood in commit messages\n55. Use present tense\n56. Add a link reference to the related story, task, or bug\nClean Commit Messages', 'section_title': 'Git Commits', 'section_type': 'Other', 'paper_id': 'TEMP_DOC', 'is_metadata': False, 'score': 0.7707622647285461, 'content': '52. Commit early & push often\n53. Write meaningful commit messages explaining the reason\n54. Use the imperative mood in commit messages\n55. Use present tense\n56. Add a link reference to the related story, task, or bug\nClean Commit Messages'}


In [10]:
sections[0].to_dict()

{'title': '',
 'content': 'From: Craft Better Software danielmoka@substack.com\nSubject: 100 Tips to Write Clean Code\nDate: 17. September 2025 at 07:04\nTo: vins23p@gmail.com\nForwarded this email? Subscribe here for more\nOm YO',
 'section_type': 'Other',
 'references_cited': [],
 'page_number': None}

**Retrieval**

In [12]:
from episcope.retrieve.embeddings import SimplifiedEmbedder

# Instantiate a simplified embedder (falls back to a stub if transformers# is unavailable).  The `dim` attribute reflects the hidden size of the# underlying model or 1 for the stub.
embedder = SimplifiedEmbedder(embed_model='distilbert-base-uncased', batch_size=2)
print(f'Embedding dimension: {embedder.dim}')

sentences = [
    'what are the important aspects for writing clean functions?',
    # 'Influenza spreads quickly in winter.',
    # 'Vaccination reduces the risk of infection.',
    # 'Personal protective equipment (PPE) helps prevent transmission.'
]
vectors = embedder.embed_texts(sentences)
for s, v in zip(sentences, vectors):
    print(f'Sentence: {s} Embedding: {v}')


No sentence-transformers model found with name distilbert-base-uncased. Creating a new one with mean pooling.


Embedding dimension: 768


100%|██████████| 1/1 [00:00<00:00,  1.99it/s]

Sentence: what are the important aspects for writing clean functions? Embedding: [0.019683051854372025, 0.007208187133073807, -0.007819120772182941, 0.005105811636894941, 0.012612758204340935, -0.02884809300303459, 0.002948711160570383, 0.02829495258629322, 0.003774044569581747, -0.019358133897185326, 0.01690319925546646, 0.008094231598079205, -0.027092624455690384, 0.007974688895046711, -0.030680611729621887, 0.012110290117561817, 0.006603736896067858, 0.01220431737601757, -0.012465103529393673, -0.0071830786764621735, -0.019086211919784546, 0.03871488571166992, -0.03294908627867699, 0.02024180069565773, 0.033143870532512665, 0.002402227371931076, 0.014546290040016174, -0.02302505448460579, -0.02803930640220642, -0.01670975610613823, -0.014147861860692501, 0.03439772129058838, -0.020074591040611267, -0.03663652762770653, -0.017169008031487465, 0.0109543576836586, 0.024547439068555832, 0.009915973991155624, -0.00029217515839263797, 0.021075593307614326, -0.06269814074039459, -0.0238965

In [13]:
from episcope.retrieve.text_retriever import TextRetriever
retriever = TextRetriever(indexer=indexer, hyde=None)

In [14]:
retriever.retrieve(query='what are the important aspects for writing clean functions?', top_k=2, use_hyde=False)

[{'text': '24. A class should have only one main responsibility\n25. Avoid large classes (~100+ lines can be a smell)\n26. Strive for one public function per class\n27. Create small private functions for single tasks\n28. Order functions based on execution flow',
  'section_title': '4 Clean Classes',
  'section_type': 'Other',
  'paper_id': 'TEMP_DOC',
  'is_metadata': False,
  'score': 0.7811035513877869,
  'content': '24. A class should have only one main responsibility\n25. Avoid large classes (~100+ lines can be a smell)\n26. Strive for one public function per class\n27. Create small private functions for single tasks\n28. Order functions based on execution flow'},
 {'text': 'Click here to Master Test-Driven Development',
  'section_title': 'CRAFT BETTER SOFTWARE',
  'section_type': 'Other',
  'paper_id': 'TEMP_DOC',
  'is_metadata': False,
  'score': 0.7710412740707397,
  'content': 'Click here to Master Test-Driven Development'}]

**Generation**

NB: the simple generator doesn t actually use the question given in input... 